In [1]:
import pandas as pd
from data_names import huc

In [2]:

#change this to your directory
base_dir = "/Users/cyber/Downloads/basin_timeseries_v1p2_metForcing_obsFlow/basin_dataset_public_v1p2"

daymet_mean_forcing_dir = f"{base_dir}/basin_mean_forcing/daymet"
basin_metadata_dir = f"{base_dir}/basin_metadata"
usgs_streamflow_dir = f"{base_dir}/usgs_streamflow"


In [3]:
#clean each individual directory and joinal

#clean daymet mean forcing - Alex
#input:  daymet_dir = path to basin_mean_forcing/daymet/
#        hucs = region folders to read, e.g. ("17",)
#output: DataFrame, one row per gauge per day
#        gauge_id (text, from the filename), date (one datetime column), weather columns
#note:   each file has 3 header lines before the column names
def clean_daymet(daymet_dir, hucs=("17",)):
    pass


In [4]:
#clean basin_metadata - JJ
#input:  metadata_dir = path to basin_metadata/
#        hucs = region codes to keep, e.g. ("17",) or ("17", "18")
#output: DataFrame, one row per gauge, 13 columns
#steps:
#1. gauge_information.txt -> huc, gauge_id, name, lat, lon, drainage area; keep only hucs
#2. basin_physical_characteristics.txt -> area, elevation, slope, forest fraction
#3. basin_annual_hydrometeorology_characteristics_daymet.txt -> long-term precip, PET, temp
#4. rename ids to gauge_id (kept as text for leading zeros) and merge all 3 on gauge_id
def clean_basin_metadata(metadata_dir, hucs=("17",)):
    #tab-separated, and gauge names contain spaces
    gauges = pd.read_csv(
        f"{metadata_dir}/gauge_information.txt",
        sep="\t",
        skiprows=1,
        header=None,
        names=["huc", "gauge_id", "gauge_name", "lat", "lon", "drainage_area_km2"],
        dtype={"huc": str, "gauge_id": str},
    )
    gauges["gauge_name"] = gauges["gauge_name"].str.strip()
    gauges = gauges[gauges["huc"].isin(hucs)]

    physical = pd.read_csv(
        f"{metadata_dir}/basin_physical_characteristics.txt",
        sep=r"\s+",
        dtype={"BASIN_ID": str},
    )
    physical = physical.drop(columns="BASIN_HUC").rename(columns={
        "BASIN_ID": "gauge_id",
        "Size(km2)": "area_km2",
        "Elevation(m)": "elevation_m",
        "Slope(m_km-1)": "slope_m_per_km",
        "Frac_Forest(percent)": "frac_forest",
    })

    #long-term climate averages; drop Runoff since it comes from streamflow (our label)
    climate = pd.read_csv(
        f"{metadata_dir}/basin_annual_hydrometeorology_characteristics_daymet.txt",
        sep=r"\s+",
        dtype={"BASIN_ID": str},
    )
    climate = climate.drop(columns=["BASIN_HUC", "Runoff(mm_day-1)"]).rename(columns={
        "BASIN_ID": "gauge_id",
        "Precip(mm_day-1)": "mean_precip_mm_day",
        "PET(mm_day-1)": "mean_pet_mm_day",
        "Temp(oC)": "mean_temp_c",
    })

    return gauges.merge(physical, on="gauge_id").merge(climate, on="gauge_id")

In [5]:
#clean usgs streamflow - Franzi
#input:  streamflow_dir = path to usgs_streamflow/
#        hucs = region folders to read, e.g. ("17",)
#output: DataFrame, one row per gauge per day
#        gauge_id (text), date (one datetime column), flow
#note:   missing flow is -999 (flag M) and must be removed
def clean_usgs_streamflow(streamflow_dir, hucs=("17",)):
    pass


In [6]:
#join all three sources - one row per gauge per day
#input:  streamflow, daymet = DataFrames with gauge_id (text) and date
#        metadata = DataFrame with one row per gauge_id
#output: DataFrame, one row per gauge per day, with flow + weather + basin facts
#steps:
#1. merge streamflow and daymet on gauge_id + date (both are daily)
#2. merge metadata on gauge_id only, so basin facts repeat on every day
#   (many_to_one errors if a gauge appears twice in metadata)
def join_data(streamflow, daymet, metadata):
    daily = streamflow.merge(daymet, on=["gauge_id", "date"])
    return daily.merge(metadata, on="gauge_id", validate="many_to_one")

In [7]:
#save the final table
#input:  df = joined DataFrame from join_data, path = output file, e.g. "pnw_dataset.csv"
#output: writes the CSV (no row index), returns nothing
#note:   read it back with pd.read_csv(path, dtype={"gauge_id": str}) to keep leading zeros
def generate_csv(df, path):
    df.to_csv(path, index=False)

In [8]:
#run the whole pipeline - change hucs here to pick regions for every step
#until clean_daymet and clean_usgs_streamflow return DataFrames, this only shows metadata
hucs = ("17",)

metadata = clean_basin_metadata(basin_metadata_dir, hucs)
daymet = clean_daymet(daymet_mean_forcing_dir, hucs)
streamflow = clean_usgs_streamflow(usgs_streamflow_dir, hucs)

if daymet is None or streamflow is None:
    print("daymet or streamflow not finished yet - showing metadata only")
    display(metadata.head())
else:
    dataset = join_data(streamflow, daymet, metadata)
    generate_csv(dataset, "pnw_dataset.csv")
    print(dataset.shape)
    display(dataset.head())

daymet or streamflow not finished yet - showing metadata only


,huc,gauge_id,gauge_name,lat,lon,drainage_area_km2,area_km2,elevation_m,slope_m_per_km,frac_forest,mean_precip_mm_day,mean_pet_mm_day,mean_temp_c
0,17,10396000,DONNER UND BLITZEN RIVER NR FRENCHGLEN OR,42.79083,-118.86750,528.89,969.57,1613.27,58.18941,0.0045,1.610,3.369,6.525
1,17,12010000,"NASELLE RIVER NEAR NASELLE, WA",46.37399,-123.74348,142.18,142.60,145.18,47.29610,0.9981,8.014,1.940,8.931
2,17,12013500,"WILLAPA RIVER NEAR WILLAPA, WA",46.65093,-123.65266,337.69,340.70,309.17,61.57144,0.9896,6.910,2.550,9.408
3,17,12020000,"CHEHALIS RIVER NEAR DOTY, WA",46.61732,-123.27764,294.22,296.74,435.98,74.62586,0.9964,7.398,2.719,8.733
4,17,12025000,"NEWAUKUM RIVER NEAR CHEHALIS, WA",46.62010,-122.94513,404.90,410.92,236.88,33.69605,0.8617,4.627,2.075,9.948
